# NumCompute quickstart

A short tour of every NumCompute module, run on a small CSV in `data/sample.csv`.
Run this notebook from the `demo/` folder.

## 1. Setup
Import NumPy and the NumCompute modules, and print the versions in use.

In [1]:
import os
import platform
import sys

# Make the package importable when it is not installed (the notebook runs from demo/).
sys.path.insert(0, os.path.abspath(".."))

import numpy as np

import numcompute
from numcompute import io, metrics, optim, rank, sort_search, stats
from numcompute.pipeline import FeatureUnion, Pipeline
from numcompute.preprocessing import ColumnSelector, OneHotEncoder, SimpleImputer, StandardScaler

np.set_printoptions(precision=3, suppress=True, linewidth=100)

print("Python     :", platform.python_version())
print("NumPy      :", np.__version__)
print("NumCompute :", numcompute.__version__)

Python     : 3.14.6
NumPy      : 2.5.4
NumCompute : 0.1.0


## 2. Read the CSV
`io.load_csv` reads the numeric columns as float (missing cells become NaN) and the city column as strings (missing cells become `""`).

In [2]:
CSV_PATH = os.path.join("data", "sample.csv")

# Read the header row as strings; the rest of the file is read below.
header = io.load_csv(CSV_PATH, dtype=str)[0].tolist()
NUM_COLS, CAT_COL = [0, 1, 2, 3], 4

X_num = io.load_csv(CSV_PATH, skip_header=1, usecols=NUM_COLS)       # float64, NaN for missing
city = io.load_csv(CSV_PATH, dtype=str, skip_header=1, usecols=[CAT_COL])  # strings, "" for missing

print("columns      :", header)
print("numeric shape:", X_num.shape, X_num.dtype)
print("city shape   :", city.shape, city.dtype)
print("first 3 rows :")
for row, c in zip(X_num[:3], city[:3, 0]):
    print("  ", row, c)

columns      : ['age', 'height_cm', 'weight_kg', 'income_k', 'city']
numeric shape: (30, 4) float64
city shape   : (30, 1) <U9
first 3 rows :
   [ 23.  172.4  68.1  41.5] Adelaide
   [ 35.  165.   59.3  62. ] Melbourne
   [ 41.  180.2  85.6   nan] Sydney


In [3]:
missing_rows = np.flatnonzero(np.isnan(X_num).any(axis=1) | (city[:, 0] == ""))
print(f"{missing_rows.size} of {len(X_num)} rows have a missing value:")
print("  row |", " | ".join(f"{h:>9}" for h in header))
for i in missing_rows:
    cells = [f"{v:9.1f}" if not np.isnan(v) else "      NaN" for v in X_num[i]]
    cells.append(f"{city[i, 0] or '<missing>':>9}")
    print(f"  {i:3d} |", " | ".join(cells))

10 of 30 rows have a missing value:
  row |       age | height_cm | weight_kg |  income_k |      city
    2 |      41.0 |     180.2 |      85.6 |       NaN |    Sydney
    4 |      52.0 |     175.9 |       NaN |      91.2 |     Perth
    6 |      26.0 |       NaN |      64.0 |      39.9 | Melbourne
    7 |      47.0 |     182.5 |      90.3 |      84.1 | <missing>
   12 |       NaN |     174.3 |      72.5 |      55.2 |  Adelaide
   13 |      33.0 |     163.8 |      58.9 |       NaN |    Sydney
   16 |      50.0 |     166.4 |       NaN |      88.0 |  Adelaide
   18 |      24.0 |     159.9 |      50.8 |      36.5 | <missing>
   22 |      28.0 |       NaN |      60.2 |      46.0 |  Adelaide
   25 |      48.0 |     178.4 |      84.5 |       NaN |    Sydney


In [4]:
# missing="skip" drops incomplete rows instead of filling them.
complete = io.load_csv(CSV_PATH, skip_header=1, usecols=NUM_COLS, missing="skip")
print("rows with complete numeric data:", complete.shape[0])

# iter_csv_chunks streams the same data in fixed-size blocks.
chunks = list(io.iter_csv_chunks(CSV_PATH, chunk_size=8, skip_header=1, usecols=NUM_COLS))
print("chunk sizes:", [len(c) for c in chunks],
      "| same as load_csv:", np.array_equal(np.vstack(chunks), X_num, equal_nan=True))

rows with complete numeric data: 22


chunk sizes: [8, 8, 8, 6] | same as load_csv: True


## 3. Preprocess
Each transformer is fitted on its own here: standardise columns, impute the remaining NaN with 0, and one-hot encode the city.

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)          # NaN is ignored when fitting and kept in the output
print("StandardScaler mean_ :", scaler.mean_)
print("StandardScaler scale_:", scaler.scale_)
print("column means after   :", np.nanmean(X_scaled, axis=0).round(6) + 0.0)
print("column stds after    :", np.nanstd(X_scaled, axis=0))

# SimpleImputer only supports strategy="constant" (mean/median are not implemented).
# After scaling, every column has mean 0, so filling with 0 means filling with the column mean.
imputer = SimpleImputer(strategy="constant", fill_value=0.0)
X_imputed = imputer.fit_transform(X_scaled)
print("\nSimpleImputer: NaN before =", int(np.isnan(X_scaled).sum()), "| NaN after =", int(np.isnan(X_imputed).sum()))
print("column means after imputing:", X_imputed.mean(axis=0).round(6) + 0.0)

StandardScaler mean_ : [ 37.379 171.489  70.411  62.63 ]
StandardScaler scale_: [10.223  6.966 11.873 19.43 ]
column means after   : [0. 0. 0. 0.]
column stds after    : [1. 1. 1. 1.]

SimpleImputer: NaN before = 8 | NaN after = 0
column means after imputing: [0. 0. 0. 0.]


In [6]:
# Turn "" into None so the encoder records it as the missing category.
city_obj = np.where(city == "", None, city).astype(object)

encoder = OneHotEncoder()
city_onehot = encoder.fit_transform(city_obj)
print("categories:", list(encoder.categories_[0]))
print("encoded shape:", city_onehot.shape)
for name, row in zip(city_obj[:8, 0], city_onehot[:8]):
    print(f"  {str(name):>9} -> {row}")

categories: ['Adelaide', 'Melbourne', 'Sydney', 'Perth', None]
encoded shape: (30, 5)
   Adelaide -> [1. 0. 0. 0. 0.]
  Melbourne -> [0. 1. 0. 0. 0.]
     Sydney -> [0. 0. 1. 0. 0.]
   Adelaide -> [1. 0. 0. 0. 0.]
      Perth -> [0. 0. 0. 1. 0.]
     Sydney -> [0. 0. 1. 0. 0.]
  Melbourne -> [0. 1. 0. 0. 0.]
       None -> [0. 0. 0. 0. 1.]


## 4. Pipeline and FeatureUnion
A `FeatureUnion` runs a numeric branch (select → scale → impute 0) and a categorical branch (select → one-hot) on the same table and joins the columns.

In [7]:
# One mixed table: 4 numeric columns plus the city column.
table = np.empty((len(X_num), 5), dtype=object)
table[:, :4] = X_num
table[:, 4] = city_obj[:, 0]

numeric_branch = Pipeline([
    ("select", ColumnSelector(NUM_COLS, dtype=float)),
    ("scale", StandardScaler()),
    ("impute", SimpleImputer(fill_value=0.0)),   # after scaling, 0 means "column mean"
])
category_branch = Pipeline([
    ("select", ColumnSelector([CAT_COL])),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
features = FeatureUnion([("numeric", numeric_branch), ("city", category_branch)])

X_features = features.fit_transform(table)
print("input shape :", table.shape)
print("output shape:", X_features.shape, "(4 scaled numeric + 5 city columns)")
print("any NaN left:", bool(np.isnan(X_features).any()))
print("first 3 rows:")
print(X_features[:3])

input shape : (30, 5)
output shape: (30, 9) (4 scaled numeric + 5 city columns)
any NaN left: False
first 3 rows:
[[-1.407  0.131 -0.195 -1.087  1.     0.     0.     0.     0.   ]
 [-0.233 -0.932 -0.936 -0.032  0.     1.     0.     0.     0.   ]
 [ 0.354  1.251  1.279  0.     0.     0.     1.     0.     0.   ]]


In [8]:
# The fitted union transforms new data without refitting; unseen cities become all zeros.
new_rows = np.array([[30, 170.0, 65.0, 50.0, "Sydney"],
                     [40, np.nan, 80.0, 70.0, "Brisbane"]], dtype=object)
print(features.transform(new_rows))

[[-0.722 -0.214 -0.456 -0.65   0.     0.     1.     0.     0.   ]
 [ 0.256  0.     0.808  0.379  0.     0.     0.     0.     0.   ]]


## 5. Sorting and searching
Top-k selection, quickselect for the k-th smallest value, and binary search on sorted data, using the age column.

In [9]:
ages = X_num[:, 0][~np.isnan(X_num[:, 0])]   # these functions reject NaN

top_vals, top_idx = sort_search.topk(ages, 5)
low_vals = sort_search.topk(ages, 3, largest=False, return_indices=False)
print("5 oldest      :", top_vals, "at positions", top_idx)
print("3 youngest    :", low_vals)

k = len(ages) // 2
print(f"quickselect k={k}:", sort_search.quickselect(ages, k),
      "| check with sort:", np.sort(ages)[k])

sorted_ages = sort_search.stable_sort(ages)
for target in (37, 40):
    idx, found = sort_search.binary_search(sorted_ages, target)
    print(f"binary_search({target}) -> index {idx}, found={found}")

5 oldest      : [58. 55. 53. 52. 50.] at positions [11 20 27  4 15]
3 youngest    : [22. 23. 24.]
quickselect k=14: 36.0 | check with sort: 36.0
binary_search(37) -> index 15, found=True
binary_search(40) -> index 18, found=False


In [10]:
# multi_key_sort: sort rows by the first column, then the second for ties.
rows = np.array([[2, 9], [1, 5], [2, 1], [1, 7]])
print(sort_search.multi_key_sort(rows, columns=[0, 1]))

[[1 5]
 [1 7]
 [2 1]
 [2 9]]


## 6. Ranking and percentiles
`rank` supports three tie rules and three NaN policies; `percentile` uses the same interpolation as `stats.quantile`.

In [11]:
scores = np.array([70, 85, 70, 90, 85, 60])
print("scores  :", scores)
for method in ("average", "dense", "ordinal"):
    print(f"{method:8}:", rank.rank(scores, method=method))

with_nan = np.array([3.0, np.nan, 1.0, 3.0])
print("\nwith NaN:", with_nan)
print("propagate:", rank.rank(with_nan))
print("omit     :", rank.rank(with_nan, nan_policy="omit"))

scores  : [70 85 70 90 85 60]
average : [2.5 4.5 2.5 6.  4.5 1. ]
dense   : [2. 3. 2. 4. 3. 1.]
ordinal : [2. 4. 3. 6. 5. 1.]

with NaN: [ 3. nan  1.  3.]
propagate: [nan nan nan nan]
omit     : [2.5 nan 1.  2.5]


In [12]:
income = X_num[:, 3]
q = [25, 50, 75, 90]
print("income percentiles", q, ":", rank.percentile(income, q, nan_policy="omit"))
for method in ("linear", "lower", "higher", "midpoint"):
    print(f"  90th, {method:8}:", rank.percentile(income, 90, method=method, nan_policy="omit"))

income percentiles [25, 50, 75, 90] : [47.15 58.8  77.85 91.76]
  90th, linear  : 91.76
  90th, lower   : 91.2
  90th, higher  : 92.6
  90th, midpoint: 91.9


## 7. Descriptive statistics
Per-column mean and variance with NaN handling, a histogram, and quantiles.

In [13]:
print("mean (propagate):", stats.mean(X_num, axis=0))
print("mean (omit)     :", stats.mean(X_num, axis=0, nan_policy="omit"))
print("variance (omit) :", stats.variance(X_num, axis=0, nan_policy="omit"))
print("std ddof=1      :", stats.std(X_num, axis=0, ddof=1, nan_policy="omit"))
print("median (omit)   :", stats.median(X_num, axis=0, nan_policy="omit"))
print("check vs NumPy  :", np.allclose(stats.variance(X_num, axis=0, nan_policy="omit"),
                                        np.nanvar(X_num, axis=0)))

mean (propagate): [nan nan nan nan]
mean (omit)     : [ 37.379 171.489  70.411  62.63 ]
variance (omit) : [104.511  48.52  140.964 377.544]
std ddof=1      : [10.404  7.093 12.091 19.801]
median (omit)   : [ 36.   171.45  70.1   58.8 ]
check vs NumPy  : True


In [14]:
counts, edges = stats.histogram(X_num[:, 0], bins=5)   # NaN omitted by default
print("age histogram:")
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"  [{lo:5.1f}, {hi:5.1f}) {'#' * c} {c}")

print("\nage quantiles [0, 0.25, 0.5, 0.75, 1]:",
      stats.quantile(X_num[:, 0], [0, 0.25, 0.5, 0.75, 1], nan_policy="omit"))

age histogram:
  [ 22.0,  29.2) ######## 8
  [ 29.2,  36.4) ####### 7
  [ 36.4,  43.6) ##### 5
  [ 43.6,  50.8) ##### 5
  [ 50.8,  58.0) #### 4

age quantiles [0, 0.25, 0.5, 0.75, 1]: [22. 29. 36. 45. 58.]


In [15]:
# StreamingStats gives the same answer from batches as from the full column.
s = stats.StreamingStats(nan_policy="omit")
for chunk in io.iter_csv_chunks(CSV_PATH, chunk_size=7, skip_header=1, usecols=[3]):
    s.update(chunk[:, 0])
print(f"streaming income: count={s.count}, mean={s.mean:.3f}, var={s.variance():.3f}")
print(f"full column     : mean={stats.mean(income, nan_policy='omit'):.3f}, "
      f"var={stats.variance(income, nan_policy='omit'):.3f}")

streaming income: count=27, mean=62.630, var=377.544
full column     : mean=62.630, var=377.544


## 8. Metrics
Classification and regression scores on small made-up predictions.

In [16]:
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0, 1, 0])

print("accuracy :", metrics.accuracy(y_true, y_pred))
print("precision:", metrics.precision(y_true, y_pred))
print("recall   :", metrics.recall(y_true, y_pred))
print("f1       :", round(metrics.f1(y_true, y_pred), 4))

cm, labels = metrics.confusion_matrix(y_true, y_pred, labels=[0, 1])
print("\nconfusion matrix (rows = true, cols = predicted), labels", list(labels))
print(cm)

accuracy : 0.8
precision: 0.8
recall   : 0.8
f1       : 0.8

confusion matrix (rows = true, cols = predicted), labels [0, 1]
[[4 1]
 [1 4]]


In [17]:
# Multi-class with string labels and different averaging modes.
t = ["cat", "dog", "bird", "cat", "dog", "bird", "cat"]
p = ["cat", "dog", "cat",  "cat", "bird", "bird", "dog"]
for avg in ("macro", "micro", "weighted"):
    print(f"{avg:8}: precision={metrics.precision(t, p, average=avg):.3f}  "
          f"recall={metrics.recall(t, p, average=avg):.3f}  f1={metrics.f1(t, p, average=avg):.3f}")

macro   : precision=0.556  recall=0.556  f1=0.556
micro   : precision=0.571  recall=0.571  f1=0.571
weighted: precision=0.571  recall=0.571  f1=0.571


In [18]:
y_reg = np.array([3.0, -0.5, 2.0, 7.0])
y_hat = np.array([2.5, 0.0, 2.0, 8.0])
print(f"MSE: {metrics.mse(y_reg, y_hat):.6f} | check: {np.mean((y_reg - y_hat) ** 2):.6f}")

MSE: 0.375000 | check: 0.375000


## 9. Numerical differentiation
Finite-difference `grad` and `jacobian` compared with the exact derivatives; central differences should be far more accurate than forward ones.

In [19]:
def f(x):
    return x[0] ** 2 * x[1] + np.sin(x[1]) + np.exp(x[0])

def f_grad_exact(x):
    return np.array([2 * x[0] * x[1] + np.exp(x[0]), x[0] ** 2 + np.cos(x[1])])

x0 = np.array([0.5, 1.2])
exact = f_grad_exact(x0)
print("exact   :", exact)
for method in ("central", "forward"):
    g = optim.grad(f, x0, method=method)
    print(f"{method:8}: {g}  max error = {np.max(np.abs(g - exact)):.1e}")

exact   : [2.849 0.612]
central : [2.849 0.612]  max error = 2.6e-11
forward : [2.849 0.612]  max error = 2.0e-05


In [20]:
def F(x):
    return np.array([x[0] * x[1], np.sin(x[0]) + x[2] ** 2, x[0] * np.exp(x[2])])

def F_jac_exact(x):
    return np.array([
        [x[1],          x[0], 0.0],
        [np.cos(x[0]),  0.0,  2 * x[2]],
        [np.exp(x[2]),  0.0,  x[0] * np.exp(x[2])],
    ])

x1 = np.array([1.0, 2.0, 0.5])
J_exact = F_jac_exact(x1)
print("jacobian shape:", optim.jacobian(F, x1).shape, "(n_outputs, n_inputs)")
for method in ("central", "forward"):
    J = optim.jacobian(F, x1, method=method)
    print(f"{method:8}: max error = {np.max(np.abs(J - J_exact)):.1e}")

jacobian shape: (3, 3) (n_outputs, n_inputs)
central : max error = 2.2e-11
forward : max error = 1.0e-05


In [21]:
# Error vs step size: forward error falls like h, central like h**2 (until rounding error dominates).
print("     h   | forward err | central err")
for h in (1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6):
    ef = np.max(np.abs(optim.grad(f, x0, method="forward", h=h) - exact))
    ec = np.max(np.abs(optim.grad(f, x0, method="central", h=h) - exact))
    print(f"  {h:.0e} |   {ef:.2e}   |   {ec:.2e}")

     h   | forward err | central err
  1e-01 |   2.05e-01   |   2.75e-03
  1e-02 |   2.03e-02   |   2.75e-05
  1e-03 |   2.02e-03   |   2.75e-07
  1e-04 |   2.02e-04   |   2.75e-09
  1e-05 |   2.02e-05   |   2.56e-11
  1e-06 |   2.02e-06   |   7.73e-11


## 10. Benchmark
> **Benchmark — waiting for benchmarking.py**
>
> `numcompute/benchmarking.py` (`benchmark`, `environment_info`, API contract section 21) does not exist yet. Replace the cell below once it lands.

In [22]:
# Benchmark — waiting for benchmarking.py
import importlib.util

if importlib.util.find_spec("numcompute.benchmarking") is None:
    print("Benchmark — waiting for benchmarking.py")
else:
    print("numcompute.benchmarking is now available: replace this placeholder with real benchmarks.")

Benchmark — waiting for benchmarking.py


## 11. Summary
- **io**: `load_csv` / `iter_csv_chunks` read numeric and string columns, with NaN or `""` for missing cells (or `missing="skip"`).
- **preprocessing**: `SimpleImputer`, `StandardScaler`, `OneHotEncoder` and `ColumnSelector` follow the `fit` / `transform` / `fit_transform` protocol.
- **pipeline**: `Pipeline` chains steps and `FeatureUnion` joins branches, turning the mixed 30×5 table into a 30×9 numeric matrix with no NaN.
- **sort_search**: `topk`, `quickselect`, `binary_search`, `stable_sort` and `multi_key_sort`.
- **rank**: average / dense / ordinal ties, NaN policies, and `percentile`.
- **stats**: NaN-aware mean, variance, std, median, `histogram`, `quantile` and `StreamingStats`.
- **metrics**: accuracy, precision, recall, F1 (binary and averaged), confusion matrix and MSE.
- **optim**: `grad` and `jacobian` match the exact derivatives; central differences are much more accurate than forward ones.
- **benchmarking**: placeholder until `numcompute/benchmarking.py` exists.